# Modelando Curvas de Demanda Não Lineares no Varejo com PROC GAM

## Resumo Executivo

Este notebook usa o **PROC GAM** para modelar as vendas semanais de unidades de
mercearia como uma função suave e não linear do preço de prateleira, da
temperatura da loja (uma variável substituta de sazonalidade) e do gasto
promocional, com um efeito paramétrico de sinalizador de promoção. Modelos
aditivos generalizados permitem que um gerente de categoria recupere as formas
curvas reais de elasticidade-preço e de demanda sazonal que uma regressão linear
deixaria passar, apoiando decisões mais afiadas de precificação e promoção.

## Fontes de Dados

| Conjunto de Dados | Linhas | Granularidade | Variáveis-Chave | Descrição |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | semana | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Série sintética semanal de ponto de venda para uma loja de mercearia principal ao longo de 100 semanas consecutivas (quase dois ciclos sazonais). Gerada inline com `call streaminit(20250531)` e `rand()`. As vendas semanais de unidades seguem um processo de demanda Poisson cuja média é impulsionada por uma curva de resposta de preço exponencial, um efeito quadrático de temperatura/sazonalidade com pico próximo a 72F, e um aumento côncavo (raiz quadrada) do gasto promocional, além de um sinalizador discreto de semana promocional. |

# Modelando Curvas de Demanda Não Lineares no Varejo com PROC GAM

A demanda no varejo raramente responde ao preço, ao clima ou ao gasto
promocional de forma linear. Um pequeno corte de preço em um produto básico
pode mal movimentar o volume, enquanto ultrapassar um ponto de preço
psicológico pode disparar um salto acentuado; a demanda impulsionada pelo clima
atinge picos em uma faixa intermediária confortável e cai nos dois extremos; e
o aumento promocional mostra retornos decrescentes conforme o gasto sobe.

O **PROC GAM** (modelos aditivos generalizados) ajusta cada driver com seu
próprio termo de spline suave, de modo que os dados — e não uma suposição
linear rígida — determinam a forma de cada curva de demanda. Aqui modelamos as
vendas semanais de unidades para uma única loja de mercearia principal ao
longo de 100 semanas consecutivas, combinando um sinalizador paramétrico de
promoção com splines de suavização em preço, temperatura e gasto promocional
sob uma resposta Poisson.

## Etapa 1 — Gerar uma série sintética de vendas semanais

Simulamos 100 semanas consecutivas (quase dois ciclos sazonais) de dados de
ponto de venda para uma loja principal. O processo gerador de dados é
deliberadamente não linear para que possamos confirmar que o GAM recupera
formas realistas:

- **Price** impulsiona o volume por meio de uma curva de resposta exponencial
  (`exp(-1.1 * Price)`), de modo que a demanda sobe acentuadamente conforme o
  preço cai.
- **Temperature** atua como uma variável substituta de sazonalidade com um pico
  quadrático próximo a 72F, modelando o tráfego motivado por conforto.
- **PromoSpend** entrega um aumento côncavo de raiz quadrada (retornos
  decrescentes).
- Um sinalizador discreto de **Promotion** adiciona um efeito paramétrico de
  degrau nas semanas promovidas.

As `Units` semanais são sorteadas de uma distribuição Poisson, condizente com a
natureza de contagem das vendas de unidades.

In [1]:
DADOS store_sales;
   CHAMAR streaminit(20250531);
   COMPRIMENTO Promotion $3;
   FAZER Week = 1 ATÉ 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      SE rand("uniform") < 0.28 ENTÃO FAZER;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      FIM;
      SENÃO FAZER;
         Promotion  = "No";
         PromoSpend = 0;
      FIM;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      SAÍDA;
   FIM;
EXECUTAR;



NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Etapa 2 — Perfilar os dados simulados

Um `PROC MEANS` rápido confirma que as variáveis abrangem faixas de varejo
sensatas antes da modelagem: as contagens de unidades são inteiros não
negativos, o preço se agrupa em torno de alguns dólares, a temperatura percorre
um ciclo sazonal completo e o gasto promocional é zero nas semanas não
promovidas.

In [2]:
PROCEDIMENTO MÉDIAS DADOS=store_sales n mean MIN MAX maxdec=2;
   VARIÁVEL Units Price Temperature PromoSpend;
EXECUTAR;


                                                  The MEANS Procedure

 Variable            N           Mean     Minimum     Maximum
 ------------------------------------------------------------
 Units             100           7.03        0.00      103.00
 Price             100           3.15        2.81        3.48
 Temperature       100          55.50       22.72       83.49
 PromoSpend        100         128.76        0.00      779.00
 ------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Etapa 3 — Ajustar o modelo aditivo completo de demanda

O modelo principal combina:

- `param(Promotion)` — um efeito paramétrico (linear) para o indicador de
  semana promocional, declarado na instrução `CLASS`.
- `spline(Price, df=4)` — uma spline cúbica de suavização capturando a resposta
  curva de preço.
- `spline(Temperature, df=4)` — uma curva suave de sazonalidade.
- `spline(PromoSpend, df=3)` — aumento promocional de retornos decrescentes.

Como as vendas de unidades são contagens, especificamos `dist=poisson` (função
de ligação log). `method=gcv` permite que a validação cruzada generalizada
guie a suavidade de qualquer componente sem uma substituição explícita de
graus de liberdade. A instrução `OUTPUT` grava previsões e resíduos por
observação em `gam_fit`.

O procedimento relata um **Deviance de 43.59** contra um **Deviance Nulo de
2041.12** — os quatro drivers suaves-mais-paramétricos explicam quase toda a
variação que um modelo apenas-constante deixaria de fora — e um **AIC de
254.61**. A estimativa paramétrica `PROMOTIONYES` é positiva (+0.41 na escala
log), confirmando o aumento de volume promocional, e a spline de preço carrega
uma tendência linear fortemente negativa (−1.71), a assinatura da demanda com
inclinação descendente.

In [3]:
PROCEDIMENTO gam DADOS=store_sales PLOTS=components(CLM commonaxes);
   CLASSE Promotion;
   MODELO Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   SAÍDA out=gam_fit predicted residual;
EXECUTAR;



                                                   The GAM Procedure                                                    

Model Information
Response Variable     Units
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Price)                    4.0000         4.


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Etapa 4 — Um modelo focado de preço e sazonalidade

Para uma revisão de precificação mais enxuta, reajustamos com apenas os dois
drivers suaves mais relevantes para a decisão — preço e temperatura — dando ao
preço flexibilidade extra (`df=5`) para resolver qualquer inflexão próxima a um
ponto de preço psicológico. O sinalizador de promoção é mantido como controle
paramétrico.

Remover o gasto promocional eleva o **Deviance para 62.80** e o **AIC para
269.82**, ambos maiores que os 43.59 e 254.61 do modelo completo. O termo
paramétrico `PROMOTIONYES` também absorve mais do sinal promocional aqui (+1.73
contra +0.41), já que a spline de gasto não está mais presente para carregá-lo.
A spline de preço mantém sua tendência negativa (−1.51), então a história
central de demanda é estável entre as especificações.

In [4]:
PROCEDIMENTO gam DADOS=store_sales PLOTS=components(CLM);
   CLASSE Promotion;
   MODELO Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
EXECUTAR;



                                                   The GAM Procedure                                                    

Model Information
Response Variable     Units
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Price)                    5.0000         5.0000
Spline(Temperature)              4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Interpretando os resultados

A tabela **Regression Model Analysis** relata a tendência linear dentro de
cada componente mais o efeito paramétrico de promoção. O coeficiente positivo
`PROMOTIONYES` (+0.41 no modelo completo, +1.73 no mais enxuto) confirma o
aumento esperado de volume promocional, enquanto a tendência linear negativa na
spline de preço (−1.71 e −1.51) reflete a clássica demanda com inclinação
descendente. O pequeno termo linear positivo da spline de temperatura (+0.03) é
esperado: sua verdadeira história é a curvatura em torno do pico de conforto de
72F, que um único coeficiente linear não consegue resumir.

A tabela **Smoothing Model Analysis** relata os graus de liberdade que cada
spline consome. Preço e temperatura consomem, cada um, 4 GL efetivos (5 para
preço no modelo mais enxuto) e o gasto promocional 3 — cada um bem acima do
único GL que uma linha reta usaria, o que explica exatamente por que uma
regressão linear deixaria passar essas relações curvas.

As **Fit Statistics** permitem comparar as duas especificações diretamente. O
modelo completo com quatro drivers registra um Deviance de 43.59 e AIC de
254.61 contra os 62.80 e 269.82 do modelo mais enxuto; ambos os critérios
favorecem o modelo completo, mostrando que o gasto promocional e o sinalizador
de promoção agregam poder explicativo além de preço e temperatura isoladamente.
Em relação ao Deviance Nulo de 2041.12, ambos os modelos capturam a
esmagadora maioria da variação de demanda.

Juntas, essas tabelas dão a um gerente de categoria uma história de demanda
quantificada e orientada por dados: uma resposta de preço acentuada que
informa a profundidade da redução, uma janela sazonal de temperatura e um
efeito de retornos decrescentes do gasto promocional — orientação muito mais
precisa do que uma única estimativa linear de elasticidade. (O PROC GAM também
aceita `plots=components` para renderizar as curvas de previsão parcial de
cada termo suave; as tabelas numéricas de componentes acima são a fonte de onde
essas curvas são desenhadas.)